# Coma-Indeterminate Group: Exploratory Structured Analysis

**Purpose.** This notebook provides the short exploratory analysis of the
coma-**indeterminate** (LLM-null / "unknown") group requested in review:

> *"If feasible, add a short exploratory analysis of this group using structured MIMIC
> data, especially sedation, ventilation, and GCS, to better explain why this group
> appears so high-risk."*

**Design principles (so the analysis is fully defensible).**
1. Every structured marker is either (a) already part of the validated pipeline
   (`GCS_Total`), (b) derived with itemids verified against `d_items.csv.gz` in this
   notebook (sedation, RASS), or (c) derived with itemids already used in
   `Preprocessing.ipynb` (ventilation, itemids 225792/225794). Nothing relies on
   unverified item codes.
2. All flags are windowed to the **first 24 h** of the ICU stay, matching the temporal
   definition used for every other structured predictor in the thesis, so the
   comparison is apples-to-apples and free of look-ahead leakage.
3. Group comparisons use the same **non-parametric, Holm-corrected** approach used
   elsewhere in the Results chapter.
4. The conclusions are read directly off the numbers; where an intuitive mechanism is
   *not* supported by the data, that is stated plainly rather than glossed over.

**Groups.** `negative` = LLM said coma absent; `positive` = LLM said coma present;
`unknown` = LLM could not commit (the indeterminate group under study).


In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)
warnings.filterwarnings('ignore')


## 0. Configuration — edit paths to match your repository

In [2]:
# --- Paths ---
MODEL_DF_PATH        = "../Outputs/model_df_final.csv"          # from structured_model_first24h.ipynb: hadm_id, llm_present_coma, GCS_Total, survival_status, baseline_risk_oof
CA_PATIENTS_FILTERED = "../Outputs/ca_patients_filtered.csv"    # from Preprocessing.ipynb: hadm_id, intime
ICU_PATH             = "../data/mimiciv/3.1/icu/"
D_ITEMS              = os.path.join(ICU_PATH, "d_items.csv.gz")
INPUTEVENTS          = os.path.join(ICU_PATH, "inputevents.csv.gz")
PROCEDUREEVENTS      = os.path.join(ICU_PATH, "procedureevents.csv.gz")
CHARTEVENTS          = os.path.join(ICU_PATH, "chartevents.csv.gz")

WINDOW_HOURS = 24  # first 24h of ICU stay, matching the structured pipeline

# --- Ventilation: itemids already used in Preprocessing.ipynb (invasive ventilation) ---
VENT_INVASIVE_ITEMIDS = {225792: "invasive_vent"}   # 225794 (non-invasive) intentionally excluded

# RASS sedation-depth score (chartevents). Verified against d_items below.
RASS_ITEMID = 228096   # "Richmond-RAS Scale"


## 1. Verify itemids against `d_items` (nothing hardcoded is trusted blindly)

The sedative and RASS itemids are resolved from the data dictionary here, by
**name search**, so the analysis uses the item codes that actually exist in this
MIMIC-IV extract rather than any assumed values.

- **Sedatives (continuous IV, `inputevents`).** We take the four standard
  sedative-hypnotics used for continuous ICU sedation — **propofol, midazolam,
  dexmedetomidine, lorazepam** — because these directly reduce level of consciousness.
  **Fentanyl is handled separately** (see below): it is an opioid analgesic that is
  frequently co-administered but does not by itself indicate depressed consciousness,
  so including it would blur the meaning of a "sedation" flag. We therefore build a
  primary sedative flag *excluding* fentanyl and report a fentanyl-inclusive version
  only as a sensitivity check.
- **RASS.** The Richmond Agitation-Sedation Scale is the clinician's direct assessment
  of consciousness/sedation depth (−5 unarousable … 0 alert … +4 combative). It is a
  more direct marker of "could this patient's neurological state be assessed?" than
  drug administration, which is why we add it.


In [3]:
d_items = pd.read_csv(D_ITEMS)

# --- Sedatives: name search restricted to inputevents (continuous/IV administration) ---
CORE_SEDATIVE_NAMES = "propofol|midazolam|dexmedetomidine|lorazepam"   # consciousness-reducing sedative-hypnotics
sed_core = d_items[
    d_items["label"].str.contains(CORE_SEDATIVE_NAMES, case=False, na=False)
    & (d_items["linksto"] == "inputevents")
]
SEDATIVE_CORE_ITEMIDS = dict(zip(sed_core["itemid"], sed_core["label"]))
print("Core sedative-hypnotic itemids (fentanyl EXCLUDED) used for the primary flag:")
print(sed_core[["itemid", "label", "unitname"]].to_string(index=False))

# Fentanyl separately, for the sensitivity flag only
fent = d_items[
    d_items["label"].str.contains("fentanyl", case=False, na=False)
    & (d_items["linksto"] == "inputevents")
]
FENTANYL_ITEMIDS = dict(zip(fent["itemid"], fent["label"]))
SEDATIVE_WITH_FENTANYL_ITEMIDS = {**SEDATIVE_CORE_ITEMIDS, **FENTANYL_ITEMIDS}
print("\nFentanyl itemids (added only in the sensitivity flag):")
print(fent[["itemid", "label", "unitname"]].to_string(index=False))

# --- RASS: confirm the itemid exists and is in chartevents ---
rass_rows = d_items[d_items["itemid"] == RASS_ITEMID]
print("\nRASS itemid check:")
print(rass_rows[["itemid", "label", "linksto"]].to_string(index=False))
rass_available_dict = len(rass_rows) > 0


Core sedative-hypnotic itemids (fentanyl EXCLUDED) used for the primary flag:
 itemid                      label unitname
 221385         Lorazepam (Ativan)       mg
 221668         Midazolam (Versed)       mg
 222168                   Propofol       mg
 225150 Dexmedetomidine (Precedex)      mcg
 229420 Dexmedetomidine (Precedex)      mcg

Fentanyl itemids (added only in the sensitivity flag):
 itemid                  label unitname
 221744               Fentanyl       mg
 225942 Fentanyl (Concentrate)       mg
 225972        Fentanyl (Push)      mcg

RASS itemid check:
 itemid              label     linksto
 228096 Richmond-RAS Scale chartevents


## 2. Load cohort and define coma groups

In [4]:
df_model = pd.read_csv(MODEL_DF_PATH)

def coma_group(x):
    if pd.isna(x):
        return "unknown"
    xs = str(x).strip().lower()
    if xs == "true":  return "positive"
    if xs == "false": return "negative"
    return "unknown"

df_model["coma_group"] = df_model["llm_present_coma"].apply(coma_group)
assert "hadm_id" in df_model.columns, "hadm_id required as the join key"

print(df_model["coma_group"].value_counts().reindex(["negative","unknown","positive"]))
print("\nObserved in-hospital mortality by coma group:")
print(df_model.groupby("coma_group")["survival_status"].mean().round(4).reindex(["negative","unknown","positive"]))


coma_group
negative    756
unknown     247
positive    615
Name: count, dtype: int64

Observed in-hospital mortality by coma group:
coma_group
negative    0.2646
unknown     0.8381
positive    0.6439
Name: survival_status, dtype: float64


## 3. First-24h flag builders (windowed, joined on `hadm_id`)

`model_df_final.csv` carries `hadm_id` but not `stay_id`, so all raw-table joins use
`hadm_id`, with `intime` taken from `ca_patients_filtered.csv`. Each marker is a binary
per-admission flag: did the event occur within the first `WINDOW_HOURS` of ICU intime.


In [5]:
def _load_hadm_intimes():
    if os.path.exists(CA_PATIENTS_FILTERED):
        s = pd.read_csv(CA_PATIENTS_FILTERED, usecols=lambda c: c in ["hadm_id", "intime"])
        if {"hadm_id", "intime"}.issubset(s.columns):
            s["intime"] = pd.to_datetime(s["intime"], errors="coerce")
            return s.dropna(subset=["hadm_id", "intime"]).drop_duplicates("hadm_id")
    if "intime" in df_model.columns:
        out = df_model[["hadm_id", "intime"]].dropna().drop_duplicates("hadm_id").copy()
        out["intime"] = pd.to_datetime(out["intime"], errors="coerce")
        return out
    return None

hadm_intimes = _load_hadm_intimes()
have_intime = hadm_intimes is not None
print(f"intime available for {0 if not have_intime else len(hadm_intimes)} admissions.")


intime available for 2307 admissions.


In [6]:
def flag_from_events(path, itemids, id_col="hadm_id", time_col="starttime",
                     usecols=None, window_h=WINDOW_HOURS):
    """Set of hadm_ids with >=1 event (itemid in itemids) within first window_h of intime."""
    if not have_intime or not itemids or not os.path.exists(path):
        if not os.path.exists(path):
            print(f"WARNING: {os.path.basename(path)} not found -> marker skipped.")
        return set()
    wanted = set(itemids)
    intimes = hadm_intimes.set_index(id_col)["intime"]
    hits = set()
    if usecols is None:
        usecols = [id_col, "itemid", time_col]
    for chunk in pd.read_csv(path, usecols=lambda c: c in usecols, chunksize=1_000_000):
        if not {"itemid", id_col, time_col}.issubset(chunk.columns):
            continue
        chunk = chunk[chunk["itemid"].isin(wanted)]
        if chunk.empty:
            continue
        chunk[time_col] = pd.to_datetime(chunk[time_col], errors="coerce")
        chunk = chunk.join(intimes, on=id_col)
        cutoff = chunk["intime"] + pd.Timedelta(hours=window_h)
        inwin = chunk[(chunk[time_col] >= chunk["intime"]) & (chunk[time_col] < cutoff)]
        hits.update(inwin[id_col].dropna().unique().tolist())
    return hits


### 3a. Sedation (primary, fentanyl-excluded) and ventilation flags

In [17]:
sed_core_hadms = flag_from_events(INPUTEVENTS, SEDATIVE_CORE_ITEMIDS,
                                  usecols=["hadm_id","itemid","starttime"])
sed_fent_hadms = flag_from_events(INPUTEVENTS, SEDATIVE_WITH_FENTANYL_ITEMIDS,
                                  usecols=["hadm_id","itemid","starttime"])
vent_hadms     = flag_from_events(PROCEDUREEVENTS, VENT_INVASIVE_ITEMIDS,
                                  usecols=["hadm_id","itemid","starttime"])

df_model["sedation_24h"]        = df_model["hadm_id"].isin(sed_core_hadms).astype(int)   # primary
df_model["sedation_fent_24h"]   = df_model["hadm_id"].isin(sed_fent_hadms).astype(int)   # sensitivity
df_model["vent_24h"]            = df_model["hadm_id"].isin(vent_hadms).astype(int)

print(f"admissions, core sedation (no fentanyl): {len(sed_core_hadms)}")
print(f"admissions, sedation incl. fentanyl:     {len(sed_fent_hadms)}")
print(f"admissions, invasive ventilation:        {len(vent_hadms)}")

admissions, core sedation (no fentanyl): 1311
admissions, sedation incl. fentanyl:     1380
admissions, invasive ventilation:        1215


### 3b. RASS (sedation depth) in the first 24 h

RASS is scored in `chartevents` with a numeric `valuenum`. We summarise two things per
admission within the first 24 h:

- **`rass_min_24h`** — the *lowest* (most sedated / least arousable) RASS recorded. A
  low minimum (e.g. −4/−5) means the patient was, at some point in the first day, too
  deeply sedated or obtunded to yield a meaningful neurological exam.
- **`rass_deep_24h`** — a binary flag for **any RASS ≤ −3** (deep sedation:
  "movement or eye opening to physical stimulation, no eye contact" or deeper). This is
  the standard threshold separating light/moderate from deep sedation.

RASS is a direct, charted assessment of consciousness depth, so it speaks more directly
to "could this patient be neurologically assessed?" than drug administration does.


In [8]:
def rass_summary(path, rass_itemid, window_h=WINDOW_HOURS):
    if not have_intime or not os.path.exists(path):
        if not os.path.exists(path):
            print(f"WARNING: {os.path.basename(path)} not found -> RASS skipped.")
        return None
    intimes = hadm_intimes.set_index("hadm_id")["intime"]
    parts = []
    usecols = ["hadm_id", "itemid", "charttime", "valuenum"]
    for chunk in pd.read_csv(path, usecols=lambda c: c in usecols, chunksize=1_000_000):
        if not {"itemid","hadm_id","charttime","valuenum"}.issubset(chunk.columns):
            continue
        chunk = chunk[chunk["itemid"] == rass_itemid]
        if chunk.empty:
            continue
        chunk["charttime"] = pd.to_datetime(chunk["charttime"], errors="coerce")
        chunk = chunk.join(intimes, on="hadm_id")
        cutoff = chunk["intime"] + pd.Timedelta(hours=window_h)
        inwin = chunk[(chunk["charttime"] >= chunk["intime"]) & (chunk["charttime"] < cutoff)]
        inwin = inwin.dropna(subset=["valuenum"])
        if not inwin.empty:
            parts.append(inwin[["hadm_id","valuenum"]])
    if not parts:
        print("WARNING: no RASS rows found in window.")
        return None
    allr = pd.concat(parts, ignore_index=True)
    agg = allr.groupby("hadm_id")["valuenum"].agg(rass_min_24h="min",
                                                  rass_median_24h="median",
                                                  rass_n_24h="count").reset_index()
    return agg

rass_df = rass_summary(CHARTEVENTS, RASS_ITEMID) if rass_available_dict else None

if rass_df is not None:
    df_model = df_model.merge(rass_df, on="hadm_id", how="left")
    # deep sedation flag: any RASS <= -3 in the window (NaN if patient never had a RASS charted)
    df_model["rass_deep_24h"] = np.where(df_model["rass_min_24h"].notna(),
                                         (df_model["rass_min_24h"] <= -3).astype(float),
                                         np.nan)
    df_model["rass_measured_24h"] = df_model["rass_min_24h"].notna().astype(int)
    print(f"admissions with >=1 RASS in first 24h: {int(df_model['rass_measured_24h'].sum())} / {len(df_model)}")
    print(df_model[["rass_min_24h","rass_median_24h","rass_deep_24h"]].describe().round(3))
else:
    for c in ["rass_min_24h","rass_median_24h","rass_deep_24h","rass_measured_24h"]:
        df_model[c] = np.nan
    print("RASS not available -> RASS columns set to NaN (analysis will skip them).")


admissions with >=1 RASS in first 24h: 1360 / 1618
       rass_min_24h  rass_median_24h  rass_deep_24h
count      1360.000         1360.000       1360.000
mean         -3.196           -2.435          0.646
std           2.034            2.146          0.479
min          -5.000           -5.000          0.000
25%          -5.000           -5.000          0.000
50%          -4.000           -2.000          1.000
75%          -1.000            0.000          1.000
max           1.000            3.000          1.000


## 4. Structured profile of the three coma groups

Rates for binary markers with Wald 95% CIs; GCS and RASS as medians with IQR.


In [9]:
def rate_ci(x):
    x = x.dropna()
    if len(x) == 0:
        return (np.nan, np.nan, np.nan, 0)
    p = x.mean(); se = np.sqrt(p*(1-p)/len(x))
    return (p, max(0,p-1.96*se), min(1,p+1.96*se), len(x))

rows = []
for g in ["negative","unknown","positive"]:
    sub = df_model[df_model["coma_group"]==g]
    row = {"group": g, "n": len(sub)}
    for flag in ["sedation_24h","sedation_fent_24h","vent_24h","rass_deep_24h"]:
        p, lo, hi, n = rate_ci(sub[flag])
        row[f"{flag}_rate"] = None if np.isnan(p) else round(p,3)
        row[f"{flag}_CI"]   = None if np.isnan(p) else f"[{lo:.3f},{hi:.3f}]"
    if "GCS_Total" in sub:
        row["GCS_med(IQR)"] = f"{sub['GCS_Total'].median():.1f} "                               f"({sub['GCS_Total'].quantile(.25):.1f}-{sub['GCS_Total'].quantile(.75):.1f})"
    if sub["rass_min_24h"].notna().any():
        row["RASSmin_med(IQR)"] = f"{sub['rass_min_24h'].median():.1f} "                                   f"({sub['rass_min_24h'].quantile(.25):.1f}-{sub['rass_min_24h'].quantile(.75):.1f})"
    row["obs_mortality"] = round(sub["survival_status"].mean(),3)
    rows.append(row)

profile = pd.DataFrame(rows)
display(profile)


,group,n,sedation_24h_rate,sedation_24h_CI,sedation_fent_24h_rate,sedation_fent_24h_CI,vent_24h_rate,vent_24h_CI,rass_deep_24h_rate,rass_deep_24h_CI,GCS_med(IQR),RASSmin_med(IQR),obs_mortality
0,negative,756,0.663,"[0.629,0.696]",0.692,"[0.659,0.725]",0.583,"[0.548,0.618]",0.526,"[0.488,0.564]",10.2 (6.9-14.7),-3.0 (-5.0-0.0),0.265
1,unknown,247,0.640,"[0.580,0.700]",0.676,"[0.618,0.734]",0.607,"[0.546,0.668]",0.665,"[0.595,0.735]",7.6 (3.3-14.0),-4.0 (-5.0--1.0),0.838
2,positive,615,0.689,"[0.653,0.726]",0.737,"[0.702,0.771]",0.686,"[0.650,0.723]",0.794,"[0.759,0.829]",4.3 (3.0-9.4),-5.0 (-5.0--3.0),0.644


## 5. Pairwise group comparisons (Holm-corrected)

Binary markers via Mann-Whitney U on the 0/1 flag (equivalent to a proportion
comparison); GCS and RASS-min via Mann-Whitney U on the continuous value. Holm
correction is applied across the three pairwise comparisons within each marker,
matching the Results-chapter convention. RASS tests use only admissions with a charted
RASS.


In [10]:
def holm_pairwise(df, group_col, value_col, groups=("negative","unknown","positive")):
    pairs = [("negative","unknown"),("unknown","positive"),("negative","positive")]
    labels, pvals, a_summ, b_summ = [], [], [], []
    for a,b in pairs:
        va = df.loc[df[group_col]==a, value_col].dropna()
        vb = df.loc[df[group_col]==b, value_col].dropna()
        if len(va)==0 or len(vb)==0:
            continue
        try:
            _, p = mannwhitneyu(va, vb, alternative="two-sided")
        except ValueError:
            p = np.nan
        labels.append(f"{a} vs {b}"); pvals.append(p)
        a_summ.append(round(va.mean(),3)); b_summ.append(round(vb.mean(),3))
    if not pvals:
        return pd.DataFrame()
    _, p_holm, _, _ = multipletests(pvals, method="holm")
    return pd.DataFrame({"marker": value_col, "comparison": labels,
                         "mean_a": a_summ, "mean_b": b_summ,
                         "p_raw": np.round(pvals,5), "p_holm": np.round(p_holm,5),
                         "sig_holm": [p < 0.05 for p in p_holm]})

for m in ["sedation_24h","vent_24h","rass_deep_24h","GCS_Total","rass_min_24h"]:
    if m in df_model.columns and df_model[m].notna().any():
        display(holm_pairwise(df_model, "coma_group", m))
    else:
        print(f"{m}: not available, skipped.")


,marker,comparison,mean_a,mean_b,p_raw,p_holm,sig_holm
0,sedation_24h,negative vs unknown,0.663,0.640,0.50845,0.58714,False
1,sedation_24h,unknown vs positive,0.640,0.689,0.15873,0.47618,False
2,sedation_24h,negative vs positive,0.663,0.689,0.29357,0.58714,False


,marker,comparison,mean_a,mean_b,p_raw,p_holm,sig_holm
0,vent_24h,negative vs unknown,0.583,0.607,0.50677,0.50677,False
1,vent_24h,unknown vs positive,0.607,0.686,0.02676,0.05351,False
2,vent_24h,negative vs positive,0.583,0.686,0.00009,0.00027,True


,marker,comparison,mean_a,mean_b,p_raw,p_holm,sig_holm
0,rass_deep_24h,negative vs unknown,0.526,0.665,0.00100,0.00103,True
1,rass_deep_24h,unknown vs positive,0.665,0.794,0.00051,0.00103,True
2,rass_deep_24h,negative vs positive,0.526,0.794,0.00000,0.00000,True


,marker,comparison,mean_a,mean_b,p_raw,p_holm,sig_holm
0,GCS_Total,negative vs unknown,10.177,8.383,0.0,0.0,True
1,GCS_Total,unknown vs positive,8.383,6.567,0.0,0.0,True
2,GCS_Total,negative vs positive,10.177,6.567,0.0,0.0,True


,marker,comparison,mean_a,mean_b,p_raw,p_holm,sig_holm
0,rass_min_24h,negative vs unknown,-2.602,-3.295,0.00003,0.00005,True
1,rass_min_24h,unknown vs positive,-3.295,-3.932,0.00002,0.00005,True
2,rass_min_24h,negative vs positive,-2.602,-3.932,0.00000,0.00000,True


## 6. GCS band distribution and the GCS-x-ventilation cross-tab

GCS bands: 3-8 (severe), 9-12 (moderate), 13-15 (mild/normal). The cross-tab within the
unknown group shows how far the low-GCS cases coincide with ventilation.


In [11]:
def gcs_bands(s):
    return pd.cut(s, bins=[2,8,12,15],
                  labels=["3-8 (severe)","9-12 (moderate)","13-15 (mild/normal)"],
                  include_lowest=True)

band = gcs_bands(df_model["GCS_Total"])
print("GCS band distribution within each coma group (row-normalised):")
display(pd.crosstab(df_model["coma_group"], band, normalize="index").round(3)
        .reindex(["negative","unknown","positive"]))

u = df_model[df_model["coma_group"]=="unknown"]
print("\nWithin coma-UNKNOWN: GCS band x ventilation (row-normalised)")
display(pd.crosstab(gcs_bands(u["GCS_Total"]), u["vent_24h"], normalize="index").round(3))


GCS band distribution within each coma group (row-normalised):


GCS_Total,3-8 (severe),9-12 (moderate),13-15 (mild/normal)
coma_group,,,
negative,0.330,0.267,0.403
unknown,0.526,0.174,0.300
positive,0.695,0.128,0.177



Within coma-UNKNOWN: GCS band x ventilation (row-normalised)


vent_24h,0,1
GCS_Total,,
3-8 (severe),0.165,0.835
9-12 (moderate),0.200,0.800
13-15 (mild/normal),0.812,0.188


## 7. Does the unknown label carry risk *beyond* GCS, sedation, and ventilation?

Logistic regression on the **negative + unknown** subset, predicting in-hospital
mortality from `GCS_Total`, `sedation_24h`, `vent_24h`, and an `unknown_ind` indicator.
The question is whether the indeterminate label remains independently associated with
mortality after adjustment.

- If `unknown_ind` stays strongly positive and significant → the group's excess risk is
  **not** explained by low GCS / sedation / ventilation; the label itself flags a
  high-risk subgroup.
- We report a version with RASS added as a sensitivity analysis (RASS restricts the
  sample to admissions with a charted RASS).


In [12]:
def fit_adjusted(df, predictors, subset_groups=("negative","unknown")):
    d = df[df["coma_group"].isin(subset_groups)].copy()
    d["unknown_ind"] = (d["coma_group"]=="unknown").astype(int)
    use_pred = [p for p in predictors if p in d.columns and d[p].notna().any()] + ["unknown_ind"]
    d = d[use_pred + ["survival_status"]].dropna()
    if d["unknown_ind"].nunique() < 2 or len(d) < 50:
        print("Insufficient data/variation to fit."); return None
    X = sm.add_constant(d[use_pred]); y = d["survival_status"]
    res = sm.Logit(y, X).fit(disp=False)
    out = pd.DataFrame({"coef": res.params.round(4),
                        "OR": np.exp(res.params).round(3),
                        "p": res.pvalues.round(4),
                        "OR_CI_low": np.exp(res.conf_int()[0]).round(3),
                        "OR_CI_high": np.exp(res.conf_int()[1]).round(3)})
    print(f"n = {len(d)}; predictors = {use_pred}")
    display(out)
    return res

print("Primary adjusted model (GCS + sedation + ventilation):")
_ = fit_adjusted(df_model, ["GCS_Total","sedation_24h","vent_24h"])

print("\nSensitivity adjusted model (adds RASS deep-sedation flag; smaller n):")
_ = fit_adjusted(df_model, ["GCS_Total","sedation_24h","vent_24h","rass_deep_24h"])


Primary adjusted model (GCS + sedation + ventilation):
n = 976; predictors = ['GCS_Total', 'sedation_24h', 'vent_24h', 'unknown_ind']


,coef,OR,p,OR_CI_low,OR_CI_high
const,0.3307,1.392,0.3607,0.685,2.829
GCS_Total,-0.0955,0.909,0.0001,0.867,0.952
sedation_24h,-0.5533,0.575,0.0190,0.362,0.913
vent_24h,-0.0931,0.911,0.6680,0.595,1.394
unknown_ind,2.6074,13.564,0.0000,9.163,20.079



Sensitivity adjusted model (adds RASS deep-sedation flag; smaller n):
n = 843; predictors = ['GCS_Total', 'sedation_24h', 'vent_24h', 'rass_deep_24h', 'unknown_ind']


,coef,OR,p,OR_CI_low,OR_CI_high
const,0.3893,1.476,0.4766,0.505,4.311
GCS_Total,-0.1008,0.904,0.0052,0.842,0.970
sedation_24h,-0.5253,0.591,0.0639,0.339,1.031
vent_24h,-0.1047,0.901,0.6873,0.541,1.499
rass_deep_24h,-0.1731,0.841,0.5756,0.459,1.542
unknown_ind,3.3192,27.638,0.0000,15.987,47.778


In [13]:
# Optional: persist the enriched columns for the thesis figures/tables
cols_out = ["hadm_id","coma_group","survival_status","GCS_Total",
            "sedation_24h","sedation_fent_24h","vent_24h",
            "rass_min_24h","rass_median_24h","rass_deep_24h","rass_measured_24h"]
cols_out = [c for c in cols_out if c in df_model.columns]
# df_model[cols_out].to_csv("../Outputs/coma_unknown_structured_profile.csv", index=False)
print("Ready to export:", cols_out)


Ready to export: ['hadm_id', 'coma_group', 'survival_status', 'GCS_Total', 'sedation_24h', 'sedation_fent_24h', 'vent_24h', 'rass_min_24h', 'rass_median_24h', 'rass_deep_24h', 'rass_measured_24h']
